## run this chromdb cell to create a new chromdb vectorbase, if the submitted one doesnt work

remove the old chromdb, instead use the this folder created by these cells, and rename to chroma_full

In [ ]:
import glob, shutil, numpy as np, pandas as pd, chromadb
from pathlib import Path
print(chromadb.__version__)

BASE = Path("data")
print(len(list((BASE / "emb_shards").glob("shard_*.npz"))), "shards found")   # expect about 23

shutil.rmtree(BASE / "chroma_full", ignore_errors=True)       # delete the broken copy

cols = ["doc_id", "ticker", "title", "text", "publish_time", "url", "embed_source", "lexrank_summary"]
meta = pd.read_parquet(BASE / "raw/cleaned_data.parquet", columns=cols).set_index("doc_id")

col = chromadb.PersistentClient(path=str(BASE / "chroma_full")).get_or_create_collection(
    "news", metadata={"hnsw:space": "cosine"})

for f in sorted((BASE / "emb_shards").glob("shard_*.npz")):
    z = np.load(f, allow_pickle=True)
    ids = [str(i) for i in z["ids"]]
    emb = z["emb"]
    p = meta.loc[ids].reset_index()
    for i in range(0, len(p), 512):
        b, eb = p.iloc[i:i+512], emb[i:i+512]
        col.upsert(
            ids=b.doc_id.tolist(), embeddings=eb.tolist(), documents=b.text.tolist(),
            metadatas=[{"ticker": r.ticker, "publish_time": int(r.publish_time),
                        "title": r.title, "url": r.url or "",
                        "embed_source": r.embed_source,
                        "lexrank_summary": r.lexrank_summary or ""}
                       for r in b.itertuples()])
    print(f.name, "| count", col.count(), flush=True)

print(col.count(), len(meta))        # should match

1.5.9
21 shards found
shard_000.npz | count 209262
shard_001.npz | count 209262
shard_002.npz | count 209262
shard_003.npz | count 209262
shard_004.npz | count 209262
shard_005.npz | count 209262
shard_006.npz | count 209262
shard_007.npz | count 209262
shard_008.npz | count 209262
shard_009.npz | count 209262
shard_010.npz | count 209262
shard_011.npz | count 209262
shard_012.npz | count 209262
shard_013.npz | count 209262
shard_014.npz | count 209262
shard_015.npz | count 209262
shard_016.npz | count 209262
shard_017.npz | count 209262
shard_018.npz | count 209262
shard_019.npz | count 209262
shard_020.npz | count 209262
209262 209262


In [3]:
import shutil, chromadb

shutil.copytree("data/chroma_full", "data/chroma_full_new")

c = chromadb.PersistentClient(path="data/chroma_full_new").get_collection("news")
print(c.count())                      # should match len(meta)
print(c.get(limit=2, include=["metadatas"])["metadatas"])

209262
[{'title': 'My 6 Largest Portfolio Holdings Heading Into 2024 -- and the Important Investing Lesson I Learned From Each One', 'url': 'https://www.nasdaq.com/articles/my-6-largest-portfolio-holdings-heading-into-2024-and-the-important-investing-lesson-i', 'embed_source': 'summary', 'ticker': 'AAPL', 'lexrank_summary': '3: Apple There\'s little question that Apple (NASDAQ: AAPL) has become one of the most successful companies in history. 6: Nvidia Every investor has one -- the "stock that got away." Should you invest $1,000 in Nvidia right now?', 'publish_time': 1702764000}, {'url': 'https://www.nasdaq.com/articles/brokers-suggest-investing-in-apple-aapl%3A-read-this-before-placing-a-bet', 'ticker': 'AAPL', 'lexrank_summary': "Brokerage Recommendation Trends for AAPL Let's take a look at what these Wall Street heavyweights have to say about Apple (AAPL) before we discuss the reliability of brokerage recommendations and how to use them to your advantage. Is AAPL a Good Investment?"

## loading all processed data files

In [1]:
%load_ext autoreload
%autoreload 2

from src.config import *
from src.loaders import load_all
from src.retrieval import retrieve, count_leaks
from src.sentiment import weekly_sentiment
from src.prices import daily_returns, history_days
from src.events import load_earnings, resolve_events
from src.event_study import event_study
from src.pipeline import run
from src.humanizer import write

ctx = load_all()
docs, sent, prices, col, model, keep = ctx.docs, ctx.sent, ctx.prices, ctx.col, ctx.model, ctx.tickers
print(docs.shape, prices.shape, col.count(), len(keep))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

docs (209262, 11) | prices (4183, 31) | chroma 209262 | 30 tickers
(209262, 11) (4183, 31) 209262 30


In [29]:
# # checking if files have been loaded properly 
# doc_id = "a57808bf082a440f"
# out = col.get(ids=[doc_id], include=["documents", "metadatas"])

# meta = out["metadatas"][0]
# print("TITLE:", meta["title"])
# print("DATE :", pd.to_datetime(meta["publish_time"], unit="s"))
# print("URL  :", meta["url"])
# print("\nARTICLE (stored, capped at 3000 chars):\n", out["documents"][0])
# print("\nLEXRANK SUMMARY:\n", meta["lexrank_summary"])

In [2]:
fb_src = docs.merge(docs[["doc_id"]], on="doc_id")   # docs already has embed_source and scores
g = docs.groupby("embed_source").agg(
    n=("sentiment", "size"),
    mean_neutral=("neu", "mean"),
    mean_abs_sentiment=("sentiment", lambda s: s.abs().mean()),
    share_neutral_dominant=("neu", lambda s: (s > 0.7).mean()))
print(g.round(3))

                   n  mean_neutral  mean_abs_sentiment  share_neutral_dominant
embed_source                                                                  
article       148279         0.433               0.455                   0.357
summary        60983         0.456               0.468                   0.387


## using openrouter to take the input text, and extract info accordign to the required schema for giving input

In [3]:
import os
from typing import Literal
import pandas as pd
from openai import OpenAI
from pydantic import BaseModel, Field
from dotenv import load_dotenv
load_dotenv()  

client = OpenAI(base_url="https://openrouter.ai/api/v1",
                api_key=os.environ["OPENROUTER_API_KEY"])      # no key in the code

class RetrievalInput(BaseModel):
    ticker: str = Field(description="Uppercase stock ticker")
    search_query: str = Field(description="Short semantic search phrase, no dates or instructions")
    start: str = Field(description="Start date, YYYY-MM-DD")
    end: str = Field(description="End date, YYYY-MM-DD")
    as_of: str = Field(description="Knowledge cutoff date, YYYY-MM-DD")
    tasks: list[Literal["trend", "event_impact", "point_lookup"]] = Field(
        description="Every task the question asks for. A question can need more than one.")
    event_hint: str = Field(description="Event phrase such as 'May earnings', or empty string if none")

SYSTEM_PROMPT = f"""Extract structured arguments from the user's financial question.
Rules:
- search_query : don't keep the company name in query, instead it should have the task the user is looking for. also dont keep the dates in it 
- ticker: map company names to tickers. Valid tickers: {', '.join(keep)}.
- Convert month or quarter ranges to exact dates ('Jan to Jun 2023' -> start 2023-01-01, end 2023-06-30).
- If the user says 'as of <date>', set as_of to that date. Otherwise set as_of equal to end.
- tasks: 'trend' for how sentiment or coverage changed over time, 'event_impact' for how the market
  reacted to an event, 'point_lookup' for what was the latest news at a date. Include all that apply.
- event_hint: the event phrase only (for example 'May earnings'); empty string if none.
- search_query: the financial topic only."""

def extract_params(user_text, model="openrouter/free"):
    schema = RetrievalInput.model_json_schema()
    schema["additionalProperties"] = False
    resp = client.chat.completions.create(
        model=model, temperature=0,
        messages=[{"role": "system", "content": SYSTEM_PROMPT},
                  {"role": "user", "content": user_text}],
        response_format={"type": "json_schema",
                         "json_schema": {"name": "retrieval_input", "strict": True, "schema": schema}},
        extra_body={"provider": {"require_parameters": True},
                    "plugins": [{"id": "response-healing"}]},
    )
    return RetrievalInput.model_validate_json(resp.choices[0].message.content)

## entire pipeline step by step

In [21]:
print(keep)

['AAPL', 'ADBE', 'ADI', 'AMAT', 'AMD', 'AMGN', 'AMZN', 'AVGO', 'BIIB', 'CMCSA', 'COST', 'CSX', 'DLTR', 'EBAY', 'GILD', 'GOOG', 'INTC', 'MSFT', 'MU', 'NVDA', 'PANW', 'PEP', 'PYPL', 'QCOM', 'SBUX', 'SIRI', 'TMUS', 'TSLA', 'TXN', 'VRTX']


In [56]:
# import pandas as pd

# # first checking if any articles are present in the given time range
# # 1. Check directly in your Parquet DataFrame (fastest, bypasses embedding filters)
# for tk in ["AAPL", "APLE"]:
#     sub = docs[
#         (docs["ticker"] == tk)
#         & (docs["publish_time"] >= ts("2023-01-01"))
#         & (docs["publish_time"] <= ts("2023-12-31") + DAY - 1)
#     ].sort_values("publish_time").copy()

#     sub["date"] = pd.to_datetime(sub["publish_time"], unit="s").dt.date
#     print(f"=== {tk}: {len(sub)} articles present ===")
#     if not sub.empty:
#         print(sub[["doc_id", "date", "embed_source", "title"]].to_string(index=False))

In [4]:
# question = ("How did sentiment around NVDA change from Jan to Jun 2023, "
#             "and how did the market react to its May earnings? Answer as of 31 May 2023.")
question = ("How did setiment around apple chenge from JAN to NOV 2023? answer as of 30 NOV 2023")
params = extract_params(question)
print(params.model_dump())

eff_end = min(pd.Timestamp(params.end), pd.Timestamp(params.as_of)).strftime("%Y-%m-%d")
print("effective end (clamped to as_of):", eff_end)

{'ticker': 'AAPL', 'search_query': 'sentiment trend', 'start': '2023-01-01', 'end': '2023-11-30', 'as_of': '2023-11-30', 'tasks': ['trend'], 'event_hint': ''}
effective end (clamped to as_of): 2023-11-30


In [5]:
res = retrieve(params.search_query, params.ticker, params.start, eff_end, params.as_of,
               col=col, model=model, sent=sent)
pd.set_option("display.max_colwidth", 75)
print(len(res), "articles")
print(res[["date", "bucket", "sim", "decay", "score", "sentiment", "title"]].round(3).to_string())
print("all on or before as_of:", res.publish_time.max() <= ts(params.as_of) + DAY - 1)

33 articles
          date  bucket    sim  decay  score  sentiment                                                                                         title
0   2023-01-03       0  0.641  0.000  0.449     -0.941                                                         Valuation Squeezes Continue into 2023
1   2023-01-04       0  0.624  0.000  0.437     -0.039                                                         Why PayPal Stock Rallied on Wednesday
2   2023-01-30       0  0.612  0.001  0.429     -0.807                                               Why Apple Is Poised For A Tough Holiday Quarter
3   2023-02-06       1  0.644  0.001  0.451      0.686                                         Has a New Bull Market Begun? (AAPL, AMZN, TSLA, META)
4   2023-02-14       1  0.632  0.001  0.443     -0.618                             US STOCKS-Wall St set for subdued open after mixed inflation data
5   2023-02-15       1  0.645  0.001  0.452      0.150                                        

In [6]:
from src.pipeline import pick_articles
arts = pick_articles(res, max_articles=15)
print(len(arts), "kept of", len(res))
for a in arts:
    print(a["date"], a["doc_id"], a["sentiment"], "ROUNDUP" if a["multi_ticker_roundup"] else "", "|", a["title"][:70])

15 kept of 33
2023-05-12 c31fa8f05da5ef67 -0.957  | US STOCKS-Indexes drop with tech-related shares; consumer sentiment fa
2023-06-19 b6d30e347262cd7c 0.042  | Bull-Bear Debate: 5 Crucial Factors to Consider in the Short-Term
2023-07-19 042ef190382356f2 0.482  | Why the Bull Market Rally Will Last Until 2025 (At Least)
2023-08-23 754be017b88405a7 -0.062  | ANALYSIS-Investors look to AI-darling Nvidia's earnings as US stocks r
2023-08-24 2e50e73ca9efc436 -0.682  | Stock Market News for Aug 24, 2023
2023-08-30 4f766be48b7b3cbe -0.943  | Stock Market News for Aug 30, 2023
2023-09-26 85534a63f071e3c1 0.642  | B of A Securities Reiterates Apple (AAPL) Neutral Recommendation
2023-09-26 23fd30d109b4ab80 -0.96  | Stocks Retreat on U.S. Economic Concerns
2023-09-27 c1a9c97f0abd4a74 -0.96  | Big Tech Titans Feel the Chill as AI Stocks Slide
2023-10-16 ca6553941d6b2cf8 0.783  | Will Big Tech Earnings Results Boost Market Sentiment?
2023-10-26 04c240458dc5fbf5 0.015  | Noteworthy Thursday Option A

In [7]:
from src.pipeline import weekly_block
weekly, wsum = weekly_block(docs, params, eff_end)
print(pd.DataFrame(weekly).to_string())
print(wsum)

    week_start  mean_sentiment  n_docs  low_confidence  partial_week
0   2022-12-26           0.016       2            True          True
1   2023-01-02          -0.262     103           False         False
2   2023-01-09          -0.016      95           False         False
3   2023-01-16          -0.067      97           False         False
4   2023-01-23          -0.091     107           False         False
5   2023-01-30          -0.286     157           False         False
6   2023-02-06          -0.086      95           False         False
7   2023-02-13          -0.067      79           False         False
8   2023-02-20          -0.135      62           False         False
9   2023-02-27          -0.030      94           False         False
10  2023-03-06          -0.132      81           False         False
11  2023-03-13          -0.095      76           False         False
12  2023-03-20           0.017      77           False         False
13  2023-03-27          -0.020    

In [8]:
events = resolve_events(prices, params.ticker, params.start, eff_end, params.as_of,
                        hint=params.event_hint, earnings=load_earnings())
print(events)

[{'date': Timestamp('2023-02-03 00:00:00'), 'kind': 'earnings'}, {'date': Timestamp('2023-05-05 00:00:00'), 'kind': 'earnings'}, {'date': Timestamp('2023-08-04 00:00:00'), 'kind': 'earnings'}, {'date': Timestamp('2023-11-03 00:00:00'), 'kind': 'earnings'}]


In [9]:
for ev in events:
    raw = event_study(prices, params.ticker, ev["date"], params.as_of)
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in raw.items()})

{'ticker': 'AAPL', 'event_date': '2023-02-03', 'status': 'ok', 'alpha': -0.0015, 'beta': 1.3489, 'sigma': 0.0118, 'n_est': 101, 'car_-1_1': 0.0504, 't_-1_1': 2.4617, 'p_-1_1': 0.0156, 'mktadj_-1_1': 0.0453, 'car_0_5': 0.0413, 't_0_5': 1.4258, 'p_0_5': 0.1571, 'mktadj_0_5': 0.0249}
{'ticker': 'AAPL', 'event_date': '2023-05-05', 'status': 'ok', 'alpha': 0.0008, 'beta': 1.3596, 'sigma': 0.0096, 'n_est': 101, 'car_-1_1': 0.0181, 't_-1_1': 1.0929, 'p_-1_1': 0.2771, 'mktadj_-1_1': 0.0249, 'car_0_5': 0.0164, 't_0_5': 0.6991, 'p_0_5': 0.4861, 'mktadj_0_5': 0.0271}
{'ticker': 'AAPL', 'event_date': '2023-08-04', 'status': 'ok', 'alpha': 0.0015, 'beta': 1.1426, 'sigma': 0.0072, 'n_est': 101, 'car_-1_1': -0.0785, 't_-1_1': -6.2591, 'p_-1_1': 0.0, 'mktadj_-1_1': -0.0738, 'car_0_5': -0.0695, 't_0_5': -3.9149, 'p_0_5': 0.0002, 'mktadj_0_5': -0.0615}
{'ticker': 'AAPL', 'event_date': '2023-11-03', 'status': 'ok', 'alpha': -0.0001, 'beta': 1.1126, 'sigma': 0.0093, 'n_est': 101, 'car_-1_1': -0.0028, 't_-

In [10]:
import json
from src.pipeline import event_record
event_recs = [event_record(prices, params.ticker, ev, params.as_of) for ev in events]
print(json.dumps(event_recs, indent=1, default=str))

[
 {
  "kind": "earnings",
  "date": "2023-02-03",
  "status": "ok",
  "beta": 1.35,
  "n_estimation_days": 101,
  "days_-1_to_+1": {
   "available": true,
   "car_pct": 5.04,
   "t_stat": 2.46,
   "p_value": 0.016,
   "market_adjusted_pct": 4.53
  },
  "days_0_to_+5": {
   "available": true,
   "car_pct": 4.13,
   "t_stat": 1.43,
   "p_value": 0.157,
   "market_adjusted_pct": 2.49
  }
 },
 {
  "kind": "earnings",
  "date": "2023-05-05",
  "status": "ok",
  "beta": 1.36,
  "n_estimation_days": 101,
  "days_-1_to_+1": {
   "available": true,
   "car_pct": 1.81,
   "t_stat": 1.09,
   "p_value": 0.277,
   "market_adjusted_pct": 2.49
  },
  "days_0_to_+5": {
   "available": true,
   "car_pct": 1.64,
   "t_stat": 0.7,
   "p_value": 0.486,
   "market_adjusted_pct": 2.71
  }
 },
 {
  "kind": "earnings",
  "date": "2023-08-04",
  "status": "ok",
  "beta": 1.14,
  "n_estimation_days": 101,
  "days_-1_to_+1": {
   "available": true,
   "car_pct": -7.85,
   "t_stat": -6.26,
   "p_value": "<0.001"

In [11]:
import json
package = {"as_of": params.as_of, "ticker": params.ticker,
           "window": {"start": params.start, "end": eff_end}, "tasks": list(params.tasks),
           "articles": arts, "weekly": weekly, "weekly_summary": wsum,
           "events": event_recs, "notes": []}
print(len(json.dumps(package, default=str)), "characters in the package")

18172 characters in the package


In [12]:
answer = write(question, package)
display(answer)

'As of 2023-11-30, sentiment around Apple became less negative from January to November 2023. In January, weekly mean sentiment was mostly below zero. The week starting 2023-01-02 averaged -0.262, and the week starting 2023-01-30 averaged -0.286 [weekly]. Other January weeks ranged from -0.016 to -0.091 [weekly].\n\nBy November, weekly mean sentiment was closer to neutral or positive. The week starting 2023-11-06 averaged -0.042, and the week starting 2023-11-13 averaged -0.036 [weekly]. The week starting 2023-11-20 averaged 0.154, while the week starting 2023-11-27 averaged 0.037, though that was a partial week [weekly].\n\nThe package’s weekly summary reports an early mean sentiment of -0.109 and a late mean sentiment of -0.004, for a change of 0.105 [weekly_summary]. Sentiment reflects the tone of news coverage, not a price movement, and this is a retrospective analysis rather than investment advice.'

In [13]:
print(answer)

As of 2023-11-30, sentiment around Apple became less negative from January to November 2023. In January, weekly mean sentiment was mostly below zero. The week starting 2023-01-02 averaged -0.262, and the week starting 2023-01-30 averaged -0.286 [weekly]. Other January weeks ranged from -0.016 to -0.091 [weekly].

By November, weekly mean sentiment was closer to neutral or positive. The week starting 2023-11-06 averaged -0.042, and the week starting 2023-11-13 averaged -0.036 [weekly]. The week starting 2023-11-20 averaged 0.154, while the week starting 2023-11-27 averaged 0.037, though that was a partial week [weekly].

The package’s weekly summary reports an early mean sentiment of -0.109 and a late mean sentiment of -0.004, for a change of 0.105 [weekly_summary]. Sentiment reflects the tone of news coverage, not a price movement, and this is a retrospective analysis rather than investment advice.


## intgrated to a single pipelien

In [42]:

def answer_question(question, show_package=True):
    params = extract_params(question)                  # 1. parse the question
    pkg = run(params, ctx)                             # 2. retrieve + sentiment + event study
    answer = write(question, pkg)           # 3. humanize
    if show_package:
        print("PARAMS :", params.model_dump())
        print("NOTES  :", pkg["notes"])
        print("WEEKLY :", json.dumps(pkg["weekly_summary"], default=str))
        print("EVENTS :", json.dumps(pkg["events"], default=str))
        print("ARTICLES:", len(pkg["articles"]), "\n")
    print(answer)
    return params, pkg, answer

In [ ]:
# question = ("How did sentiment around NVDA change from Jan to Jun 2023, "
#             "and how did the market react to its May earnings? Answer as of 31 May 2023.")
params, pkg, answer = answer_question(question)


PARAMS : {'ticker': 'NVDA', 'search_query': 'NVDA sentiment and market reaction to May earnings', 'start': '2023-01-01', 'end': '2023-06-30', 'as_of': '2023-05-31', 'tasks': ['trend', 'event_impact'], 'event_hint': 'May earnings'}
NOTES  : ['Window end 2023-06-30 was cut to the as-of date 2023-05-31. Nothing after it is used.', 'Event day 0 is the first trading day the market could react. After-close announcements count from the next trading day.']
WEEKLY : {"n_articles_total": 1495, "overall_mean_sentiment": 0.079, "busiest_week": {"week_start": "2023-05-22", "n_docs": 225}, "peak_week": {"week_start": "2023-04-10", "mean_sentiment": 0.227}, "trough_week": {"week_start": "2023-01-02", "mean_sentiment": -0.135}, "weeks_in_early_and_late_groups": 4, "early_mean_sentiment": -0.012, "late_mean_sentiment": 0.133, "change_late_minus_early": 0.144}
EVENTS : [{"kind": "earnings", "date": "2023-05-25", "status": "ok", "beta": 2.03, "n_estimation_days": 101, "days_-1_to_+1": {"available": true,

## comparing with asking a standard llm

In [14]:
def ask_plain(question, model="openrouter/free", system=None):
    msgs = ([{"role": "system", "content": system}] if system else []) + \
           [{"role": "user", "content": question}]
    r = client.chat.completions.create(model=model, temperature=0, messages=msgs)
    return r.choices[0].message.content.strip()

plain = ask_plain(question)
print(plain)

**Short answer:**  
As of 30 Nov 2023 the overall “sentiment” (i.e., the net emotional tone) surrounding Apple’s official channels – its press‑release pages, X/Twitter feed, Instagram, YouTube videos, and the Apple Newsroom blog – has been **moderately positive**, with a few noticeable dips tied to privacy‑related news and occasional competition.  

Below is a more detailed breakdown of what the conversation looked like from **January 1 2023 → November 30 2023**.

---

## 1. What “sentiment around Apple channel” means

| Term | Typical interpretation |
|------|-----------------------|
| **Channel** | Apple’s own communication outlets: Apple.com/newsroom, Apple’s X (Twitter) account, Instagram, YouTube “Apple” channel, and the “What’s New” posts on the website. |
| **Sentiment** | The aggregate emotional polarity expressed by users, journalists, analysts, and the broader public when they read or discuss those posts. It is usually captured as a mix of **positive (+), neutral (0), and neg

In [44]:
px = prices["NVDA"]; r = px.pct_change()
print(px.loc["2023-01-03"], px.loc["2023-05-24"], px.loc["2023-05-25"])
print("May 24:", round(r.loc["2023-05-24"], 4), "| May 25:", round(r.loc["2023-05-25"], 4))

14.267297744750977 30.441408157348633 37.85987091064453
May 24: -0.0049 | May 25: 0.2437


In [25]:
prices

Ticker,AAPL,ADBE,ADI,AMAT,AMD,AMGN,AMZN,AVGO,BIIB,CMCSA,...,PEP,PYPL,QCOM,SBUX,SIRI,TMUS,TSLA,TXN,VRTX,^GSPC
Date,,,,,,,,,,,,,,,,,,,,,
2008-06-02,5.561958,43.459999,22.670586,14.498049,6.800000,29.343779,4.011500,NaN,61.410000,6.899422,...,38.562515,NaN,30.646416,6.537129,19.637108,32.299316,NaN,20.038214,28.549999,1385.670044
2008-06-03,5.540139,43.529999,22.878633,14.125348,6.810000,29.297331,4.005500,NaN,61.709999,6.822828,...,38.277084,NaN,30.190250,6.471502,19.869049,32.204777,NaN,19.695099,29.530001,1377.650024
2008-06-04,5.534759,44.250000,23.320730,14.341513,7.190000,29.529589,4.075000,NaN,62.250000,6.933122,...,38.302937,NaN,31.128286,6.606403,20.564850,32.740467,NaN,19.944637,28.850000,1377.199951
2008-06-05,5.661481,43.889999,23.424761,14.386236,7.780000,29.814926,4.225500,NaN,64.570000,6.960695,...,38.659126,NaN,31.732224,6.752240,20.874102,33.544018,NaN,19.982059,32.630001,1404.050049
2008-06-06,5.548208,42.299999,22.625082,14.080622,7.430000,29.357063,4.031500,NaN,62.419998,6.721728,...,37.642223,NaN,30.286629,6.442336,19.869049,32.582920,NaN,19.489222,32.240002,1360.680054
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-01-07,240.282379,422.630005,210.485306,175.168488,127.330002,247.281418,222.110001,225.350433,153.149994,31.869944,...,135.682587,86.999786,154.588425,88.303818,21.150784,205.766998,394.359985,182.196777,408.320007,5909.029785
2025-01-08,240.768478,419.579987,214.941284,174.350861,121.839996,249.310150,222.130005,226.010788,150.750000,32.129253,...,136.746399,86.970116,153.201630,88.075546,19.731575,208.848572,394.940002,181.675400,412.109985,5918.250000
2025-01-10,234.965027,405.920013,207.072601,169.090485,116.040001,247.441833,218.940002,221.082733,148.820007,31.429102,...,133.107025,82.440636,151.179214,87.742653,19.072332,204.759216,394.739990,180.244064,409.559998,5827.040039
